# Sparse Reconstruction Visualization (pycolmap + hloc)

This notebook follows the `hloc.utils.viz_3d` workflow:
- load reconstruction with `pycolmap`
- optionally search `models/` and pick the largest registered model
- visualize points + camera poses using `plot_reconstruction`

In [ ]:
from pathlib import Path

# Change only this dataset name:
DATASET = "RSRD-10"

# Visualization parameters (same style as hloc example)
HEIGHT = 800
MIN_TRACK_LENGTH = 2
MAX_REPROJ_ERROR = 6.0
CAMERA_SIZE = 0.4
SHOW_POINTS = True
SHOW_CAMERAS = True
POINTS_RGB = True

cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    repo_root = cwd
elif (cwd.parent / "data").exists():
    repo_root = cwd.parent
else:
    repo_root = cwd

sfm_dir = repo_root / "data" / DATASET / "sparse" / "0"
print(f"repo_root: {repo_root}")
print(f"sfm_dir  : {sfm_dir}")
assert sfm_dir.exists(), f"SFM dir not found: {sfm_dir}"

repo_root: /home/scymz2/JNeRF
sfm_dir  : /home/scymz2/JNeRF/hloc_data/RSRD-1/sparse/0


In [5]:
import pycolmap
from hloc.utils.viz_3d import init_figure, plot_reconstruction

def value_or_call(obj):
    return obj() if callable(obj) else obj

def count_reg_images(rec):
    if hasattr(rec, "num_reg_images"):
        try:
            return int(value_or_call(rec.num_reg_images))
        except Exception:
            pass
    if hasattr(rec, "reg_image_ids"):
        try:
            return len(list(value_or_call(rec.reg_image_ids)))
        except Exception:
            pass
    return 0

# Search the largest model in sfm_dir/models/* first.
models_dir = sfm_dir / "models"
best_model_path = sfm_dir
best_num_images = -1

if models_dir.exists():
    for model_path in sorted(models_dir.iterdir()):
        if not model_path.is_dir():
            continue
        try:
            rec_tmp = pycolmap.Reconstruction(str(model_path))
            n_reg = count_reg_images(rec_tmp)
            if n_reg > best_num_images:
                best_num_images = n_reg
                best_model_path = model_path
        except Exception:
            pass

if best_num_images < 0:
    # Fallback: use sfm_dir directly
    rec_tmp = pycolmap.Reconstruction(str(sfm_dir))
    best_num_images = count_reg_images(rec_tmp)
    best_model_path = sfm_dir

print(f"Loading model from: {best_model_path}")

Loading model from: /home/scymz2/JNeRF/hloc_data/RSRD-1/sparse/0


In [6]:
rec = pycolmap.Reconstruction(str(best_model_path))

num_reg = count_reg_images(rec)
num_points = len(rec.points3D)
print(f"Registered images: {num_reg}")
print(f"3D points: {num_points}")

fig = init_figure(height=HEIGHT)
plot_reconstruction(
    fig,
    rec,
    min_track_length=MIN_TRACK_LENGTH,
    max_reproj_error=MAX_REPROJ_ERROR,
    cs=CAMERA_SIZE,
    points=SHOW_POINTS,
    cameras=SHOW_CAMERAS,
    points_rgb=POINTS_RGB,
)
fig.show()

Registered images: 4
3D points: 342
